# Fine-tune QWEN in Google Colab

This notebook guide provides a comprehensive overview of using the `transformers` Python package to efficiently train a custom model. It covers the following techniques:

1. Utilizing model, tokenizer, and dataset loading functionalities from Hugging Face.
2. Performing basic data cleaning.
3. Training the model with basic modeling techniques, including quantization, such as qlora in this instance.
4. Evaluating the model's performance on test set.
5. Saving your custom model and preparing it for deployment.

## Preliminary Preparation

Before proceeding with model training, ensure your environment is properly configured by following these steps:

1. Install the necessary Python packages.
2. Import the required libraries.

In [ ]:
!pip install  h5py typing-extensions wheel fschat
# !pip install -q accelerate==0.21.0 peft==0.4.0 bitsandbytes==0.40.2 transformers==4.31.0 fschat
!pip install  -U bitsandbytes
!pip install  -U git+https://github.com/huggingface/transformers.git
!pip install  -U git+https://github.com/huggingface/peft.git
!pip install  -U git+https://github.com/huggingface/accelerate.git
!pip install  datasets

In [ ]:
!nvidia-smi

## Load Pre-trained model and tokenizer

First let's load the model we are going to use - phoenix-inst-chat-7b! Note that the model itself is around 7B in full precision

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Quantization type (fp4 or nf4), According to QLoRA paper, for training 4-bit base models (e.g. using LoRA adapters) one should use
bnb_4bit_quant_type = "nf4"

# Activate nested quantization for 4-bit base models (double quantization)
use_nested_quant = True

model_id = "C:/Users/Administrator/.cache/huggingface/hub/Qwen3-4B-Instruct-2507"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=use_nested_quant,
    bnb_4bit_quant_type=bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map={"":0})

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Then we have to apply some preprocessing to the model to prepare it for training. For that use the `prepare_model_for_kbit_training` method from PEFT.

In [2]:
from peft import prepare_model_for_kbit_training

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

In [3]:
def print_trainable_parameters(model):
    """
    Prints the number of trainable parameters in the model.
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}"
    )

In [4]:
from peft import LoraConfig, get_peft_model
# You can try differnt parameter-effient strategy for model trianing, for more info, please check https://github.com/huggingface/peft
config = LoraConfig(
    r=8,
    lora_alpha=8,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, config)

In [5]:
from fastchat.conversation import get_conv_template
device = "cuda"
model.eval()

@torch.no_grad()
def generate(prompt):
    input_ids = tokenizer.encode(prompt, add_special_tokens=False, return_tensors='pt').to(device)
    outputs = model.generate(input_ids, do_sample=False, max_new_tokens=1024)
    return tokenizer.decode(*outputs, skip_special_tokens=True)

prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
response = generate(text)
print("-"*80)
print(response)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


--------------------------------------------------------------------------------
user
Give me a short introduction to large language model.
assistant
A large language model (LLM) is a type of artificial intelligence system trained on vast amounts of text from the internet to understand and generate human-like language. These models use deep learning techniques, particularly transformer architectures, to learn patterns, grammar, and context in natural language. Once trained, LLMs can perform a wide range of tasks such as answering questions, writing stories, translating languages, summarizing text, and more—often producing responses that are coherent and contextually relevant. Examples include GPT-3, GPT-4, and Llama. While powerful, they can sometimes produce inaccurate or biased information and should be used with critical thinking and verification.


## Data Preparation

Let's load a common dataset, english quotes, to fine tune our model on famous quotes.

In [6]:
from datasets import load_dataset

# data = load_dataset("Abirate/english_quotes")
dataset = load_dataset("FreedomIntelligence/Huatuo26M-Lite")
dataset = dataset['train'].map(lambda sample: {"conversations": [{"role": "human", "value": sample['question']}, {"role": "gpt", "value": sample['answer']}]}, batched=False)

Using the latest cached version of the dataset since FreedomIntelligence/Huatuo26M-Lite couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at C:\Users\Administrator\.cache\huggingface\datasets\FreedomIntelligence___huatuo26_m-lite\default\0.0.0\90ce61699e90568db82cdce4c4035c6918d70aa6 (last modified on Tue Nov 11 23:02:24 2025).


In [7]:
from torch.utils.data import random_split

In [8]:
train_dataset_size, val_dataset_size = 50, 10
train_dataset, val_dataset, _ = random_split(dataset, [train_dataset_size, val_dataset_size, len(dataset)-train_dataset_size-val_dataset_size])

### Customized Dataset
Create a specialized dataset class named "InstructionDataset" designed to handle our custom dataset.

In [10]:
import json, copy
import transformers
from typing import Dict, Sequence, List
from dataclasses import dataclass
from torch.utils.data import Dataset

IGNORE_INDEX = -100
DEFAULT_PAD_TOKEN = "<pad>"
DEFAULT_BOS_TOKEN = "<s>"
DEFAULT_EOS_TOKEN = "</s>"
DEFAULT_UNK_TOKEN = "<unk>"
default_conversation = get_conv_template('phoenix')

class InstructDataset(Dataset):
    def __init__(self, data: Sequence, tokenizer: transformers.PreTrainedTokenizer) -> None:
        super().__init__()
        self.tokenizer = tokenizer
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index) -> Dict[str, torch.Tensor]:
        sources = self.data[index]
        if isinstance(index, int):
            sources = [sources]
        data_dict = preprocess([e['conversations'] for e in sources], self.tokenizer)
        if isinstance(index, int):
            data_dict = dict(input_ids=data_dict["input_ids"][0], labels=data_dict["labels"][0])
        return data_dict

def preprocess(
        sources: Sequence[str],
        tokenizer: transformers.PreTrainedTokenizer,
        max_length=1024
) -> Dict:
    # add end signal and concatenate together
    conversations = []
    intermediates = []
    for source in sources:
        header = f"{default_conversation.system_message}"
        conversation, intermediate = _add_speaker_and_signal(header, source)
        conversations.append(conversation)
        intermediates.append(intermediate)

    # tokenize conversations
    conversations_tokenized = _tokenize_fn(conversations, tokenizer)
    input_ids = conversations_tokenized["input_ids"]
    targets = copy.deepcopy(input_ids)

    # keep only machine responses as targets
    assert len(targets) == len(intermediates)
    for target, inters in zip(targets, intermediates):
        mask = torch.zeros_like(target, dtype=torch.bool)
        for inter in inters:
            tokenized = _tokenize_fn(inter, tokenizer)
            start_idx = tokenized["input_ids"][0].size(0) - 1
            end_idx = tokenized["input_ids"][1].size(0)
            mask[start_idx:end_idx] = True
        target[~mask] = IGNORE_INDEX

    input_ids = input_ids[:max_length]
    targets = targets[:max_length]
    return dict(input_ids=input_ids, labels=targets)

def _add_speaker_and_signal(header, source, get_conversation=True):
    BEGIN_SIGNAL = DEFAULT_BOS_TOKEN
    END_SIGNAL = DEFAULT_EOS_TOKEN
    conversation = header
    intermediate = []
    for sentence in source:
        from_str = sentence["role"]
        if from_str.lower() == "human":
            from_str = default_conversation.roles[0]
        elif from_str.lower() == "gpt":
            from_str = default_conversation.roles[1]
        else:
            from_str = 'unknown'
        # store the string w/o and w/ the response
        value = (from_str + ": " + BEGIN_SIGNAL + sentence["value"] + END_SIGNAL)
        if sentence["role"].lower() == "gpt":
            start = conversation + from_str + ": " + BEGIN_SIGNAL
            end = conversation + value
            intermediate.append([start, end])
        if get_conversation:
            conversation += value
    return conversation, intermediate

def _tokenize_fn(strings: Sequence[str], tokenizer: transformers.PreTrainedTokenizer) -> Dict:
    tokenized_list = [
        tokenizer(
            text,
            return_tensors="pt",
            padding="longest",
            max_length=tokenizer.model_max_length,
            truncation=True,
        ) for text in strings
    ]
    input_ids = labels = [tokenized.input_ids[0] for tokenized in tokenized_list]
    input_ids_lens = labels_lens = [
        tokenized.input_ids.ne(tokenizer.pad_token_id).sum().item()
        for tokenized in tokenized_list
    ]
    return dict(
        input_ids=input_ids,
        labels=labels,
        input_ids_lens=input_ids_lens,
        labels_lens=labels_lens,
    )

@dataclass
class DataCollatorForSupervisedDataset(object):
    tokenizer: transformers.PreTrainedTokenizer

    def __call__(self, instances: Sequence[Dict]) -> Dict[str, torch.Tensor]:
        input_ids, labels = tuple([instance[key] for instance in instances] for key in ("input_ids", "labels"))
        input_ids = torch.nn.utils.rnn.pad_sequence(
            input_ids,
            batch_first=True,
            padding_value=self.tokenizer.pad_token_id)
        labels = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=IGNORE_INDEX)
        return dict(
            input_ids=input_ids,
            labels=labels,
            attention_mask=input_ids.ne(self.tokenizer.pad_token_id),
        )

In [11]:
train_dataset = InstructDataset(train_dataset, tokenizer)
val_dataset = InstructDataset(val_dataset, tokenizer)
data_collator = DataCollatorForSupervisedDataset(tokenizer=tokenizer)

In [12]:
sample_data = train_dataset[1]

print("=" * 80)
print("Debuging: ")
print(sample_data)
print("-" * 80)
print(f"input_ids:\n{tokenizer.decode(sample_data['input_ids'])}")
# Filter out IGNORE_INDEX before decoding labels
z = [token for token in sample_data['labels'] if token != IGNORE_INDEX]
print("-" * 80)
print(f"labels:\n{tokenizer.decode(z)}")
print("=" * 80)

Debuging: 
{'input_ids': tensor([    32,   6236,   1948,    264,  22208,   3738,    323,    458,  20443,
         11229,  17847,     13,    576,  17847,   6696,  10950,     11,  11682,
            11,    323,  47787,  11253,    311,    279,   3738,    594,   4755,
           382,  33975,     25,    366,     82,     29, 102512, 100205, 100226,
         57218, 102512,  82075, 100226,  20412, 102410,  91572, 102670,   1773,
         18493, 104459, 110652,  13343,  73670, 107272,  34230, 103876, 102512,
         99180, 106256,  99566,  99375,  29490,  99789, 106256,   1773, 114173,
         99471,  91572, 107272, 101037,    522,     82,     29,  71703,     25,
           366,     82,     29,  34230, 103876, 102512,  99180, 106256,  33108,
         99566,  99375,  29490,  99789, 106256, 100132, 102870,  23384, 100067,
          3837,  73670,  91572, 107272,   3837,  77288,  85106,  18493, 103998,
          9370, 109767,  71817,  37132,     82,     29]), 'labels': tensor([  -100,   -100,   -

## Training

### General Training Hyperparameters

In [19]:
# Set training parameters
training_arguments = transformers.TrainingArguments(
    output_dir="./checkpoints",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    optim='paged_adamw_32bit',
    eval_strategy="steps",         # <--- 新增: 按步数进行验证
    eval_steps=10,                       # <--- 新增: 每10步验证一次
    save_steps=10,                       # 建议与eval_steps保持一致
    logging_steps=10,                    # 建议与eval_steps保持一致
    learning_rate=2e-5,
    weight_decay=0.001,
    max_steps=-1,
    warmup_ratio=0.03,
    bf16=True,
    group_by_length=True,
    lr_scheduler_type="cosine",
    gradient_checkpointing=True,
    report_to="none"
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
model.train()
trainer = transformers.Trainer(
    model=model,
    args=training_arguments,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

In [ ]:
model.print_trainable_parameters()

In [ ]:
trainer.train()

Once the training is completed, we can evaluate our model and get its perplexity on the validation set like this:

In [ ]:
import matplotlib.pyplot as plt

# 从trainer.state.log_history中提取日志
log_history = trainer.state.log_history

# 初始化用于存储损失值的列表
train_steps = []
train_losses = []
val_steps = []
val_losses = []

# 遍历日志历史
for log in log_history:
    # 检查是否是训练日志
    if 'loss' in log:
        train_steps.append(log['step'])
        train_losses.append(log['loss'])
    # 检查是否是验证日志
    if 'eval_loss' in log:
        val_steps.append(log['step'])
        val_losses.append(log['eval_loss'])

# 开始绘图
plt.figure(figsize=(12, 6))

# 绘制训练损失曲线
plt.plot(train_steps, train_losses, label='Training Loss', color='blue', marker='o', linestyle='-')

# 绘制验证损失曲线
plt.plot(val_steps, val_losses, label='Validation Loss', color='red', marker='x', linestyle='--')

# 添加图表标题和坐标轴标签
plt.title('Training and Validation Loss Curves', fontsize=16)
plt.xlabel('Steps', fontsize=12)
plt.ylabel('Loss', fontsize=12)

# 添加图例
plt.legend()

# 添加网格
plt.grid(True)

# 显示图表
plt.show()


In [ ]:
import math
eval_results = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

## Save Trained LoRA

In [ ]:
output_path = "lora"
trainer.save_model(output_path)

### Test the trained model

In [ ]:
from fastchat.conversation import get_conv_template
device = "cuda"
model.eval()
@torch.no_grad()
def generate(prompt):
    input_ids = tokenizer.encode(prompt, add_special_tokens=False, return_tensors='pt').to(device)
    outputs = trainer.model.generate(input_ids, do_sample=False, max_new_tokens=1024)
    return tokenizer.decode(*outputs, skip_special_tokens=True)

prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
response = generate(text)
print("-"*80)
print(response)

# Clean GPU Memory

In [ ]:
# Empty VRAM
del model
del trainer
import gc
gc.collect()
gc.collect()

## Load the trained model back and integrate the trained LoRA within.

In [ ]:
from peft import PeftModel
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Re-define quantization parameters if needed, or use the ones defined earlier
bnb_4bit_quant_type = "nf4" # or "fp4"
use_nested_quant = True # or False

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=use_nested_quant,
    bnb_4bit_quant_type=bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map={"":0})
model = PeftModel.from_pretrained(model, output_path)
model = model.merge_and_unload()
model.config.max_length = 512
model.eval()

tokenizer = AutoTokenizer.from_pretrained(model_id, padding_side="left")
# tokenizer.pad_token = tokenizer.unk_token # This line is not needed and can be removed

## Answer generation

In [ ]:
from tqdm import tqdm
@torch.no_grad()
def generate(query_list, return_answer: bool = False):
    def conv_format(query):
        conv = get_conv_template('phoenix')
        conv.append_message(conv.roles[0], query)
        conv.append_message(conv.roles[1], None)
        return conv.get_prompt()
    query_list = [conv_format(query) for query in query_list]
    input_ids = tokenizer(query_list, padding=True, truncation=True, return_tensors="pt", add_special_tokens=False).input_ids.to("cuda")
    n_input, n_seq = input_ids.shape[0], input_ids.shape[-1]
    output_ids = []
    step = 1
    for index in tqdm(range(0, n_input, step)):
        outputs = model.generate(
            input_ids=input_ids[index: min(n_input, index+step)],
            do_sample=False,
            max_new_tokens=64,
            # temperature=0.7,
            repetition_penalty=1.0,
        )
        output_ids += outputs
    responses = tokenizer.batch_decode(output_ids, skip_special_tokens=True)
    if return_answer:
        return [response[len(query):].strip() for query, response in zip(query_list, responses)]
    return responses

# test
print("\n".join(generate(["What's the weather like today?", "Who are you?"])))


## Evaluate a trained model on a given test dataset

In [ ]:
import os
# TODO: correctly put test data files into an accessible path
test_file = "zh_med.json"
assert os.path.exists(test_file), "Invalid test_file path"

with open(test_file, 'r', encoding='utf-8') as reader:
    test_data = json.load(reader)
print(test_data[0])

In [ ]:
model_answers = generate([data[0] for data in test_data], return_answer=True)

In [ ]:
for data, answer in zip(test_data, model_answers):
    data.append(answer)

In [ ]:
with open("saved_data.json", 'w', encoding='utf-8') as writer:
    json.dump(test_data, writer, indent=4, ensure_ascii=False)